# 112 — De modelo y automatización a agente

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Definición operativa:** agente = LLM que, en un bucle, decide qué acción ejecutar
(incluida terminar), observa el resultado real y usa esa observación para el siguiente
paso, al servicio de un objetivo verificable y bajo límites explícitos. Cuatro componentes
necesarios: objetivo, acciones, observación, bucle de decisión.

**Tres regímenes:**

- **Modelo:** una llamada, sin entorno.
- **Workflow:** grafo de pasos escrito por el ingeniero; el LLM rellena casillas.
- **Agente:** el control de flujo lo decide el modelo en cada iteración; la trayectoria
  emerge de la interacción con el entorno.

**Regla de ingeniería** (Anthropic, *Building effective agents*): usar la solución más
simple que resuelva la tarea. Agente solo cuando pasos y orden no se conocen a priori.


### 🎚️ Espectro de autonomía

```text
L0 modelo puro → L1 workflow con LLM → L2 router → L3 agente acotado (solo lectura
o con aprobación) → L4 agente con efectos + presupuesto/permisos → L5 autonomía extendida
```

A mayor autonomía, más superficie de fallo y más controles obligatorios.

El laboratorio `agent` ejecuta el caso mínimo: objetivo "verificar estado y sumar 7 + 5",
dos herramientas (`status`, `sum`), y una traza donde cada acción conserva argumentos y
observación. Su limitación declarada — "el plan es determinista" — señala la diferencia
con un agente LLM: aquí la política está cableada; allá la elige el modelo cada iteración.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Contrato del laboratorio.** Ejecuta `run_lab("agent", seed=112)` y
verifica que el resultado incluya `kind`, `evidence` y `limitations`. Reconstruye a partir
de `result.trace` el estado del objetivo tras cada paso: ¿qué condición verificó cada
acción y en qué momento el agente pudo terminar en éxito?

**Ejercicio 2 — Clasifica los sistemas.** Ubica cada sistema en el espectro L0-L5 y
justifica en una línea: (a) un prompt que traduce un correo; (b) un pipeline fijo
extraer → clasificar → responder con tres llamadas al modelo; (c) un LLM que elige entre
`buscar_web` y `responder_directo` según la pregunta; (d) un asistente de código que
edita archivos, ejecuta tests en un bucle hasta que pasan, con máximo de 20 iteraciones;
(e) el mismo asistente pero sin límite de iteraciones ni permisos.

**Ejercicio 3 — Objetivo verificable.** El objetivo "mejora la documentación del
proyecto" no es un predicado verificable. Reescríbelo como objetivo operativo (condición
de éxito medible sobre el entorno + condición de parada), y señala qué podría "optimizar
literalmente" un agente con la versión ambigua.

**Ejercicio 4 — Traza contrafactual.** Escribe a mano la traza (acción → observación →
decisión) que produciría el agente del laboratorio si la herramienta `status()` fallara en
el primer intento con `{"error": "timeout"}` y funcionara al segundo. ¿Cuántos pasos
consume y qué debería registrar la traza para que un auditor entienda el reintento?


In [ ]:
# TODO: ejecuta run_lab("agent", seed=112)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 1: reconstruye el estado del objetivo paso a paso
result = run_lab("agent", seed=112)
objetivo = {"healthy": None, "sum": None}
for paso in result["result"]["trace"]:
    accion = paso["action"]
    obs = paso["observation"]
    # completa: actualiza 'objetivo' según la herramienta invocada
    # y decide si el agente ya puede terminar
    pass
print(objetivo)


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 4: traza contrafactual con reintento
traza_contrafactual = [
    # {"action": {...}, "observation": {...}, "decision": "..."},
]
# ¿cuántos pasos consume? ¿qué campo añadirías para auditar el reintento?


## Reflexión

1. La traza del laboratorio termina cuando `healthy == true` y `sum == 12`. ¿Qué haría el
   bucle si `status()` devolviera `healthy: false`, y por qué esa diferencia (parar por
   observación vs parar por "ya ejecuté mis pasos") es la frontera entre agente y script?
2. El laboratorio declara "el plan es determinista" como limitación. ¿Qué dos propiedades
   nuevas (una de capacidad, una de riesgo) aparecen cuando la política de decisión pasa
   de estar cableada a ser elegida por un LLM en cada iteración?
3. Da un ejemplo de tarea de tu entorno que HOY resolverías como workflow y el cambio de
   requisito mínimo que la convertiría en un problema de agente. ¿Qué límite (presupuesto,
   permiso o aprobación) añadirías en ese mismo momento?
